In [1]:
import pandas as pd
import numpy as np


In [2]:
file_path = "/content/IKEA SALES DATA.xlsx"

In [3]:
df = pd.read_excel(file_path)

In [5]:
# Keep only actual transaction records.
# This excludes the helper calculations below the table.
df = df[df["OrderID"].notna()].copy()

In [6]:
# Reset row numbering
df.reset_index(drop=True, inplace=True)

In [7]:
# Inspect the dataset
print("Dataset shape:", df.shape)

Dataset shape: (1025, 30)


In [8]:
print("first 5 rows ")
display(df.head(5))

first 5 rows 


,OrderID,OrderDate,ShipDate,CustomerID,CustomerName,Region,City,ProductCategory,ProductSubCategory,ProductName,...,Rating,Month,Year,DeliveryDate,Feedback,Supplier,Country,CostPrice,Margin,Tax
0,O1001,2024-09-15,2024-09-18,CUST-1001,Lars Anderson,North America,New York,Furniture,Sofas,KIVIK Sofa,...,5.0,9.0,2024.0,2024-09-20,Excellent,Inter IKEA Supply AG,USA,450.0,0.3271,60.72
1,O1002,2024-10-01,2024-10-01,CUST-1002,Maria Rodriguez,Europe,Madrid,Storage,Shelving Unit,KALLAX Shelf,...,4.0,10.0,2024.0,2024-10-01,Positive,Swedwood,Spain,50.0,0.4717,22.89
2,O1003,2024-10-05,2024-10-09,CUST-1003,Kenji Tanaka,Asia,Tokyo,Kitchen,Dinnerware,OFTAST Plate,...,3.0,10.0,2024.0,2024-10-10,Neutral,Local Supplier JP,Japan,6.5,0.4916,1.68
3,O1004,2024-11-20,2024-11-20,CUST-1004,Fatima Al Marzooqi,Middle East,Dubai,Textiles,Bedding,GULLKLOCKA Duvet,...,1.0,11.0,2024.0,2024-11-20,Negative,IKEA Industry,UAE,85.0,-0.1878,10.00
4,O1005,2024-12-10,2024-12-11,CUST-1005,David Chen,North America,Toronto,Lighting,Lamps,FADO Table Lamp,...,5.0,12.0,2024.0,2024-12-12,Excellent,IKEA Industry,Canada,40.0,0.4097,7.20


In [9]:

print("\nColumn information:")
df.info()


Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   OrderID             1025 non-null   object        
 1   OrderDate           1025 non-null   datetime64[ns]
 2   ShipDate            1025 non-null   datetime64[ns]
 3   CustomerID          1025 non-null   object        
 4   CustomerName        1025 non-null   object        
 5   Region              1025 non-null   object        
 6   City                1025 non-null   object        
 7   ProductCategory     1025 non-null   object        
 8   ProductSubCategory  1025 non-null   object        
 9   ProductName         1025 non-null   object        
 10  Quantity            1025 non-null   float64       
 11  UnitPrice           1025 non-null   object        
 12  Discount            1025 non-null   float64       
 13  Sales               1025 no

In [10]:
print("\nMissing values:")
print(df.isnull().sum().sum())


Missing values:
0


In [11]:
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate OrderIDs:", df["OrderID"].duplicated().sum())


Duplicate rows: 0
Duplicate OrderIDs: 0


INVESTIGATINF DATA TYPES AND VALUES


In [12]:

# STEP 3: Investigate suspicious data types and values

# 1. Inspect raw UnitPrice values
print("First 20 UnitPrice values:")
print(df["UnitPrice"].head(20).tolist())

print("\nUnitPrice data type:")
print(df["UnitPrice"].dtype)

print("\nUnique UnitPrice values:")
print(df["UnitPrice"].nunique())

# 2. Test which UnitPrice values can be converted to numbers
unit_test = pd.to_numeric(
    df["UnitPrice"]
      .astype("string")
      .str.replace(r"[$₹€£,\s]", "", regex=True),
    errors="coerce"
)

invalid_prices = df.loc[
    unit_test.isna(),
    ["OrderID", "UnitPrice"]
]

print("\nUnitPrice values that fail numeric conversion:")
print(invalid_prices.to_string(index=False))

print("\nNumber of invalid prices:", len(invalid_prices))

# 3. Inspect Quantity and Discount
print("\nQuantity summary:")
print(df["Quantity"].describe())

print("\nDiscount summary:")
print(df["Discount"].describe())

print("\nUnique discount values:")
print(sorted(df["Discount"].dropna().unique()))

# 4. Inspect Sales, Profit, CostPrice and Tax
print("\nFinancial columns summary:")
display(
    df[["Sales", "Profit", "CostPrice", "Tax"]].describe()
)


First 20 UnitPrice values:
[799, 45.99, 1.49, 49.99, 19.99, 546.04, 957.28, 773.41, 271.04, 140.91, 448.33, 737.91, 260.12, 930.56, 1.89, 993.45, 11.2, 884.09, 86.99, 59.99]

UnitPrice data type:
object

Unique UnitPrice values:
123

UnitPrice values that fail numeric conversion:
Empty DataFrame
Columns: [OrderID, UnitPrice]
Index: []

Number of invalid prices: 0

Quantity summary:
count    1025.000000
mean        5.712195
std         3.604752
min         1.000000
25%         3.000000
50%         5.000000
75%         8.000000
max        15.000000
Name: Quantity, dtype: float64

Discount summary:
count    1025.000000
mean        0.053415
std         0.064988
min         0.000000
25%         0.000000
50%         0.020000
75%         0.100000
max         0.300000
Name: Discount, dtype: float64

Unique discount values:
[np.float64(0.0), np.float64(0.01), np.float64(0.02), np.float64(0.03), np.float64(0.04), np.float64(0.05), np.float64(0.06), np.float64(0.07), np.float64(0.08), np.float64(

,Sales,Profit,CostPrice,Tax
count,1025.000000,1025.000000,1025.000000,1025.000000
mean,77167.559307,35420.462976,8123.527522,9472.346400
std,116714.019625,56920.544082,10507.686651,14381.976918
min,1.890000,-37742.600000,0.850000,0.020000
25%,650.000000,185.060000,93.240000,60.720000
50%,2980.350000,845.000000,507.990000,298.040000
75%,143118.730000,60443.430000,16239.500000,17300.220000
max,471246.300000,264848.510000,56445.890000,62580.460000


In [13]:

# STEP 4: Validate order, shipping and delivery dates

print("ShipDate before OrderDate:")
print((df["ShipDate"] < df["OrderDate"]).sum())

print("\nDeliveryDate before OrderDate:")
print((df["DeliveryDate"] < df["OrderDate"]).sum())

print("\nDeliveryDate before ShipDate:")
print((df["DeliveryDate"] < df["ShipDate"]).sum())

# Recalculate Month and Year from the actual order date
calculated_month = df["OrderDate"].dt.month
calculated_year = df["OrderDate"].dt.year

print("\nMonth mismatches:")
print((df["Month"] != calculated_month).sum())

print("\nYear mismatches:")
print((df["Year"] != calculated_year).sum())


ShipDate before OrderDate:
0

DeliveryDate before OrderDate:
85

DeliveryDate before ShipDate:
87

Month mismatches:
0

Year mismatches:
0


In [14]:

# STEP 5: Create a working copy without changing the original

df_raw = df.copy(deep=True)
df_work = df_raw.copy(deep=True)

# 1. Convert UnitPrice to numeric
df_work["UnitPrice"] = pd.to_numeric(
    df_work["UnitPrice"], errors="coerce"
)

print("Invalid UnitPrice values:",
      df_work["UnitPrice"].isna().sum())

# 2. Check whether quantities are whole numbers
quantity_is_whole = np.isclose(
    df_work["Quantity"],
    df_work["Quantity"].round()
).all()

print("Are all quantities whole numbers?",
      quantity_is_whole)

# 3. Flag invalid date sequences
df_work["flag_delivery_before_order"] = (
    df_work["DeliveryDate"] < df_work["OrderDate"]
)

df_work["flag_delivery_before_ship"] = (
    df_work["DeliveryDate"] < df_work["ShipDate"]
)

# 4. Recalculate expected sales for validation
df_work["ExpectedSales"] = (
    df_work["Quantity"]
    * df_work["UnitPrice"]
    * (1 - df_work["Discount"])
).round(2)

df_work["SalesDifference"] = (
    df_work["Sales"] - df_work["ExpectedSales"]
).round(2)

df_work["flag_sales_mismatch"] = (
    df_work["SalesDifference"].abs() > 0.02
)

# 5. Check margin representation
profit_sales_ratio = (
    df_work["Profit"] / df_work["Sales"].replace(0, np.nan)
)

margin_as_fraction = np.isclose(
    df_work["Margin"], profit_sales_ratio,
    atol=0.002
)

margin_as_percentage = np.isclose(
    df_work["Margin"], profit_sales_ratio * 100,
    atol=0.02
)

# 6. Print audit summary
print("\n--- DATA QUALITY AUDIT ---")
print("Rows:", len(df_work))
print("Columns:", len(df_work.columns))

print("Invalid date sequences (delivery before order):",
      df_work["flag_delivery_before_order"].sum())

print("Invalid date sequences (delivery before shipping):",
      df_work["flag_delivery_before_ship"].sum())

print("Sales formula mismatches:",
      df_work["flag_sales_mismatch"].sum())

print("Margin matches fraction scale:",
      margin_as_fraction.sum())

print("Margin matches percentage scale:",
      margin_as_percentage.sum())

# 7. Inspect records with the largest sales discrepancies
audit_columns = [
    "OrderID", "OrderDate", "ProductName",
    "Quantity", "UnitPrice", "Discount",
    "Sales", "ExpectedSales", "SalesDifference"
]

display(
    df_work.loc[
        df_work["flag_sales_mismatch"], audit_columns
    ]
    .assign(
        AbsoluteDifference=lambda x:
            x["SalesDifference"].abs()
    )
    .sort_values("AbsoluteDifference", ascending=False)
    .head(15)
)

# 8. Inspect invalid delivery dates
display(
    df_work.loc[
        df_work["flag_delivery_before_ship"],
        [
            "OrderID", "OrderDate", "ShipDate",
            "DeliveryDate", "ProductName"
        ]
    ].head(15)
)


Invalid UnitPrice values: 0
Are all quantities whole numbers? True

--- DATA QUALITY AUDIT ---
Rows: 1025
Columns: 35
Invalid date sequences (delivery before order): 85
Invalid date sequences (delivery before shipping): 87
Sales formula mismatches: 274
Margin matches fraction scale: 5
Margin matches percentage scale: 932


,OrderID,OrderDate,ProductName,Quantity,UnitPrice,Discount,Sales,ExpectedSales,SalesDifference,AbsoluteDifference
678,O1679,2024-09-08,Storage Model-S1,8.0,44275.64,0.17,274154.51,293990.25,-19835.74,19835.74
928,O1929,2025-01-29,Bookcases Model-961,8.0,44607.72,0.18,295593.65,292626.64,2967.01,2967.01
694,O1695,2025-01-16,Paper Model-P1,9.0,22810.02,0.01,205047.88,203237.28,1810.60,1810.60
973,O1974,2024-08-01,Bookcases Model-106,6.0,41285.83,0.06,234552.24,232852.08,1700.16,1700.16
853,O1854,2024-08-01,Bookcases Model-106,6.0,47596.26,0.06,269894.49,268442.91,1451.58,1451.58
603,O1604,2024-09-08,Storage Model-S1,9.0,47124.63,0.06,400118.86,398674.37,1444.49,1444.49
826,O1827,2024-09-08,Art Model-R1,10.0,48821.57,0.12,430932.61,429629.82,1302.79,1302.79
716,O1717,2024-01-09,Phones Model-880,9.0,41285.83,0.03,361667.62,360425.30,1242.32,1242.32
1006,O2007,2024-09-08,Art Model-R1,9.0,39281.42,0.15,299295.66,300502.86,-1207.20,1207.20
745,O1746,2024-08-01,Bookcases Model-106,6.0,47124.63,0.06,266948.51,265782.91,1165.60,1165.60


,OrderID,OrderDate,ShipDate,DeliveryDate,ProductName
6,O1007,2024-08-26,2024-08-28,2024-08-27,TILLREDA Portable Induction Hob
103,O1104,2023-04-13,2023-04-18,2023-04-16,STOENSE Rug
587,O1588,2025-01-29,2025-02-06,2025-01-08,Accessories Model-A1
600,O1601,2025-01-16,2025-01-20,2025-01-04,Phones Model-120
611,O1612,2025-01-20,2025-01-24,2025-01-04,Chairs Model-803
612,O1613,2025-04-18,2025-04-20,2025-01-02,Art Model-R1
613,O1614,2025-04-21,2025-04-28,2025-01-07,Phones Model-880
616,O1617,2025-04-28,2025-05-08,2025-01-10,Binders Model-B1
619,O1620,2025-02-14,2025-02-24,2025-01-10,Phones Model-880
634,O1635,2025-01-16,2025-01-22,2025-01-06,Tables Model-260


In [15]:

# STEP 6: Identify the two product groups

df_work["ProductGroup"] = np.where(
    df_work["ProductName"]
    .astype("string")
    .str.contains(
        r"\bModel-",
        case=False,
        regex=True,
        na=False
    ),
    "Generic model-labelled products",
    "IKEA-style named products"
)

# Compare data quality by group
group_summary = (
    df_work.groupby("ProductGroup")
    .agg(
        Records=("OrderID", "count"),
        UniqueProducts=("ProductName", "nunique"),
        AverageUnitPrice=("UnitPrice", "mean"),
        SalesMismatches=("flag_sales_mismatch", "sum"),
        DeliveryBeforeOrder=(
            "flag_delivery_before_order", "sum"
        ),
        DeliveryBeforeShip=(
            "flag_delivery_before_ship", "sum"
        )
    )
    .round(2)
)

display(group_summary)

# Create separate working datasets.
# Keep both groups; do not delete or overwrite the source.
df_ikea_candidate = df_work[
    df_work["ProductGroup"] == "IKEA-style named products"
].copy()

df_generic_review = df_work[
    df_work["ProductGroup"] == "Generic model-labelled products"
].copy()

print("IKEA-style candidate rows:", len(df_ikea_candidate))
print("Generic model-labelled rows:", len(df_generic_review))

# Inspect the transition between the groups
boundary_orders = ["O1584", "O1585", "O1586", "O1587", "O1588"]

boundary_columns = [
    "OrderID", "OrderDate", "ProductName",
    "UnitPrice", "Sales", "Profit",
    "CustomerID", "Region", "City", "Country"
]

display(
    df_work.loc[
        df_work["OrderID"].isin(boundary_orders),
        boundary_columns
    ]
)


,Records,UniqueProducts,AverageUnitPrice,SalesMismatches,DeliveryBeforeOrder,DeliveryBeforeShip
ProductGroup,,,,,,
Generic model-labelled products,440,19,34494.78,255,85,85
IKEA-style named products,585,25,365.09,19,0,2


IKEA-style candidate rows: 585
Generic model-labelled rows: 440


,OrderID,OrderDate,ProductName,UnitPrice,Sales,Profit,CustomerID,Region,City,Country
583,O1584,2024-09-30,HEMNES Storage,86.99,695.92,277.57,CUST914,Asia,Mumbai,India
584,O1585,2023-03-20,LINNMON/ADILS Desk,260.12,1040.48,423.85,CUST552,Oceania,Sydney,Australia
585,O1586,2024-11-20,Accessories Model-A2,22810.02,164232.14,26084.77,CUST874,East,Hyderabad,India
586,O1587,2024-04-09,Chairs Model-803,21808.99,151523.51,-17616.50,CUST368,West,Delhi,India
587,O1588,2025-01-29,Accessories Model-A1,38829.39,113524.47,43285.39,CUST830,East,Chennai,USA


In [16]:

# STEP 7: Prepare the IKEA-style candidate dataset

# Start from the untouched source, not the audit-enriched DataFrame
clean_df = df_raw.loc[
    ~df_raw["ProductName"]
    .astype("string")
    .str.contains(
        r"\bModel-",
        case=False,
        regex=True,
        na=False
    )
].copy()

# 1. Standardise whitespace in text columns
text_cols = clean_df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_cols:
    clean_df[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
    )

# 2. Convert UnitPrice to a numeric data type
clean_df["UnitPrice"] = pd.to_numeric(
    clean_df["UnitPrice"],
    errors="coerce"
)

# 3. Ensure date columns are proper datetime values
date_cols = ["OrderDate", "ShipDate", "DeliveryDate"]

for col in date_cols:
    clean_df[col] = pd.to_datetime(
        clean_df[col],
        errors="coerce"
    )

# 4. Recalculate Month and Year from OrderDate
clean_df["Month"] = clean_df["OrderDate"].dt.month
clean_df["Year"] = clean_df["OrderDate"].dt.year

# 5. Flag invalid date sequences
clean_df["flag_delivery_before_order"] = (
    clean_df["DeliveryDate"] < clean_df["OrderDate"]
)

clean_df["flag_delivery_before_ship"] = (
    clean_df["DeliveryDate"] < clean_df["ShipDate"]
)

# 6. Validate Sales without changing the original Sales column
clean_df["ExpectedSales"] = (
    clean_df["Quantity"]
    * clean_df["UnitPrice"]
    * (1 - clean_df["Discount"])
).round(2)

clean_df["SalesDifference"] = (
    clean_df["Sales"] - clean_df["ExpectedSales"]
)

clean_df["flag_sales_mismatch"] = (
    clean_df["SalesDifference"].abs() > 0.02
)

# 7. Calculate a consistent profit margin measure
# This assumes Margin means Profit / Sales.
clean_df["CalculatedMarginPct"] = (
    clean_df["Profit"]
    .div(clean_df["Sales"].replace(0, np.nan))
    .mul(100)
)

# 8. Flag customer IDs associated with multiple names
customer_name_counts = (
    clean_df.groupby("CustomerID")["CustomerName"]
    .nunique()
)

clean_df["flag_customer_id_conflict"] = (
    clean_df["CustomerID"]
    .map(customer_name_counts)
    .gt(1)
)

# 9. Summarise the validation results
print("CLEANING PREPARATION SUMMARY")
print("----------------------------")
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("Missing values in original columns:",
      clean_df[df_raw.columns].isna().sum().sum())

print("\nUnitPrice dtype:", clean_df["UnitPrice"].dtype)
print("Invalid date sequences:",
      clean_df["flag_delivery_before_order"].sum(),
      "delivery dates before order dates")
print("Deliveries before shipping:",
      clean_df["flag_delivery_before_ship"].sum())
print("Sales mismatches:",
      clean_df["flag_sales_mismatch"].sum())
print("Rows affected by customer ID conflicts:",
      clean_df["flag_customer_id_conflict"].sum())


CLEANING PREPARATION SUMMARY
----------------------------
Rows: 585
Columns: 37
Missing values in original columns: 0

UnitPrice dtype: Float64
Invalid date sequences: 0 delivery dates before order dates
Deliveries before shipping: 2
Sales mismatches: 19
Rows affected by customer ID conflicts: 567


In [17]:

# STEP 8A: Investigate conflicting CustomerIDs

customer_conflicts = (
    clean_df.groupby("CustomerID")
    .agg(
        RecordCount=("OrderID", "size"),
        UniqueNames=("CustomerName", "nunique"),
        CustomerNames=(
            "CustomerName",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
    .query("UniqueNames > 1")
    .sort_values(
        ["UniqueNames", "RecordCount"],
        ascending=False
    )
)

print("Number of conflicting CustomerIDs:",
      len(customer_conflicts))

print("\nMost problematic CustomerIDs:")
display(customer_conflicts.head(20))

# Check the reverse relationship:
# Does one customer name appear under multiple IDs?

name_conflicts = (
    clean_df.groupby("CustomerName")
    .agg(
        RecordCount=("OrderID", "size"),
        UniqueIDs=("CustomerID", "nunique"),
        CustomerIDs=(
            "CustomerID",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
    .query("UniqueIDs > 1")
    .sort_values("UniqueIDs", ascending=False)
)

print("\nCustomer names linked to multiple IDs:",
      len(name_conflicts))

display(name_conflicts.head(15))


Number of conflicting CustomerIDs: 46

Most problematic CustomerIDs:


,RecordCount,UniqueNames,CustomerNames
CustomerID,,,
CUST991,13,9,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST777,30,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST755,24,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST754,21,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST914,18,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST666,17,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST444,16,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST521,16,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."
CUST883,15,8,"[David Chen, Emily White, Fatima Al Marzooqi, ..."



Customer names linked to multiple IDs: 11


,RecordCount,UniqueIDs,CustomerIDs
CustomerName,,,
Emily White,77,38,"[CUST101, CUST130, CUST131, CUST168, CUST180, ..."
Lars Anderson,71,38,"[CUST-1001, CUST101, CUST130, CUST131, CUST168..."
David Chen,77,36,"[CUST-1005, CUST130, CUST180, CUST212, CUST240..."
Maria Rodriguez,75,35,"[CUST-1002, CUST130, CUST210, CUST211, CUST212..."
Kenji Tanaka,76,35,"[CUST-1003, CUST131, CUST180, CUST211, CUST222..."
John Smith,78,35,"[CUST101, CUST130, CUST168, CUST180, CUST211, ..."
Fatima Al Marzooqi,71,33,"[CUST-1004, CUST111, CUST130, CUST131, CUST211..."
Priya Sharma,17,16,"[CUST101, CUST112, CUST131, CUST240, CUST312, ..."
Michael Brown,20,15,"[CUST101, CUST130, CUST211, CUST316, CUST355, ..."


In [18]:

# STEP 8B: Investigate sales calculation differences

# Formula 1: Quantity x UnitPrice x (1 - Discount)
discounted_sales = (
    clean_df["Quantity"]
    * clean_df["UnitPrice"]
    * (1 - clean_df["Discount"])
).round(2)

# Formula 2: Quantity x UnitPrice, without discount
undiscounted_sales = (
    clean_df["Quantity"]
    * clean_df["UnitPrice"]
).round(2)

sales_comparison = clean_df[
    [
        "OrderID", "ProductName", "Quantity",
        "UnitPrice", "Discount", "Sales", "Profit"
    ]
].copy()

sales_comparison["ExpectedDiscountedSales"] = discounted_sales
sales_comparison["ExpectedUndiscountedSales"] = undiscounted_sales

sales_comparison["DiscountedDifference"] = (
    sales_comparison["Sales"]
    - sales_comparison["ExpectedDiscountedSales"]
).round(2)

sales_comparison["UndiscountedDifference"] = (
    sales_comparison["Sales"]
    - sales_comparison["ExpectedUndiscountedSales"]
).round(2)

# Compare formulas across the full candidate dataset
print(
    "Matches discounted formula:",
    (sales_comparison["DiscountedDifference"].abs() <= 0.02).sum()
)

print(
    "Matches undiscounted formula:",
    (sales_comparison["UndiscountedDifference"].abs() <= 0.02).sum()
)

# Display the mismatching transactions
display(
    sales_comparison.loc[
        sales_comparison["DiscountedDifference"].abs() > 0.02
    ]
    .assign(
        AbsoluteDifference=lambda x:
            x["DiscountedDifference"].abs()
    )
    .sort_values("AbsoluteDifference", ascending=False)
    .drop(columns="AbsoluteDifference")
)


Matches discounted formula: 566
Matches undiscounted formula: 425


,OrderID,ProductName,Quantity,UnitPrice,Discount,Sales,Profit,ExpectedDiscountedSales,ExpectedUndiscountedSales,DiscountedDifference,UndiscountedDifference
162,O1163,ODGER Chair,11.0,448.33,0.05,4662.63,1774.22,4685.05,4931.63,-22.42,-269.0
499,O1500,ODGER Chair,6.0,448.33,0.10,2410.98,917.47,2420.98,2689.98,-10.0,-279.0
125,O1126,BILLY Bookcase,7.0,271.04,0.05,1795.73,647.93,1802.42,1897.28,-6.69,-101.55
439,O1440,HEKTAR Floor Lamp,5.0,930.56,0.05,4417.06,1618.36,4420.16,4652.8,-3.1,-235.74
66,O1067,BILLY Bookcase,6.0,271.04,0.20,1297.90,487.68,1300.99,1626.24,-3.09,-328.34
400,O1401,GULLKLOCKA Duvet,7.0,59.99,0.05,401.95,65.55,398.93,419.93,3.02,-17.98
517,O1518,STOENSE Rug,7.0,140.91,0.05,938.83,360.85,937.05,986.37,1.78,-47.54
277,O1278,STOENSE Rug,7.0,140.91,0.05,938.83,360.85,937.05,986.37,1.78,-47.54
354,O1355,STOENSE Rug,5.0,140.91,0.05,670.73,275.50,669.32,704.55,1.41,-33.82
225,O1226,HEKTAR Floor Lamp,3.0,930.56,0.05,2651.09,954.39,2652.1,2791.68,-1.01,-140.59


In [19]:

# STEP 8C: Inspect delivery dates that precede shipping

delivery_issues = clean_df.loc[
    clean_df["DeliveryDate"] < clean_df["ShipDate"],
    [
        "OrderID", "OrderDate", "ShipDate",
        "DeliveryDate", "ShipMode", "ProductName"
    ]
]

display(delivery_issues)


,OrderID,OrderDate,ShipDate,DeliveryDate,ShipMode,ProductName
6,O1007,2024-08-26,2024-08-28,2024-08-27,Same Day,TILLREDA Portable Induction Hob
103,O1104,2023-04-13,2023-04-18,2023-04-16,First Class,STOENSE Rug


In [20]:

# STEP 9: Financial consistency audit

audit = clean_df[
    [
        "OrderID", "Quantity", "UnitPrice", "Discount",
        "Sales", "Profit", "CostPrice", "Margin", "Tax"
    ]
].copy()

# Candidate sales formulas
audit["ExpectedDiscountedSales"] = (
    audit["Quantity"]
    * audit["UnitPrice"]
    * (1 - audit["Discount"])
).round(2)

audit["ExpectedUndiscountedSales"] = (
    audit["Quantity"] * audit["UnitPrice"]
).round(2)

# Candidate profit formulas
audit["ProfitFromMarginPct"] = (
    audit["Sales"] * audit["Margin"] / 100
).round(2)

audit["ProfitFromMarginFraction"] = (
    audit["Sales"] * audit["Margin"]
).round(2)

audit["ProfitFromSalesCostTax"] = (
    audit["Sales"] - audit["CostPrice"] - audit["Tax"]
).round(2)

audit["ProfitFromQuantityCostTax"] = (
    audit["Sales"]
    - audit["Quantity"] * audit["CostPrice"]
    - audit["Tax"]
).round(2)

# Compare each candidate against recorded values
tolerance = 0.02

checks = {
    "Discounted sales": (
        audit["Sales"] - audit["ExpectedDiscountedSales"]
    ).abs() <= tolerance,

    "Undiscounted sales": (
        audit["Sales"] - audit["ExpectedUndiscountedSales"]
    ).abs() <= tolerance,

    "Profit matches percentage margin": (
        audit["Profit"] - audit["ProfitFromMarginPct"]
    ).abs() <= tolerance,

    "Profit matches fractional margin": (
        audit["Profit"] - audit["ProfitFromMarginFraction"]
    ).abs() <= tolerance,

    "Profit = Sales - CostPrice - Tax": (
        audit["Profit"] - audit["ProfitFromSalesCostTax"]
    ).abs() <= tolerance,

    "Profit = Sales - Quantity x CostPrice - Tax": (
        audit["Profit"] - audit["ProfitFromQuantityCostTax"]
    ).abs() <= tolerance
}

print("FINANCIAL VALIDATION SUMMARY")
print("----------------------------")

for rule, matches in checks.items():
    print(f"{rule}: {matches.sum()} / {len(audit)}")

print("\nRows with the 19 known sales discrepancies:")
display(
    audit.loc[
        (
            audit["Sales"] - audit["ExpectedDiscountedSales"]
        ).abs() > tolerance
    ].sort_values(
        by="OrderID"
    )
)

print("\nDiscount distribution:")
print(
    "Zero discount:",
    (audit["Discount"] == 0).sum()
)
print(
    "Non-zero discount:",
    (audit["Discount"] != 0).sum()
)


FINANCIAL VALIDATION SUMMARY
----------------------------
Discounted sales: 566 / 585
Undiscounted sales: 425 / 585
Profit matches percentage margin: 333 / 585
Profit matches fractional margin: 4 / 585
Profit = Sales - CostPrice - Tax: 20 / 585
Profit = Sales - Quantity x CostPrice - Tax: 31 / 585

Rows with the 19 known sales discrepancies:


,OrderID,Quantity,UnitPrice,Discount,Sales,Profit,CostPrice,Margin,Tax,ExpectedDiscountedSales,ExpectedUndiscountedSales,ProfitFromMarginPct,ProfitFromMarginFraction,ProfitFromSalesCostTax,ProfitFromQuantityCostTax
41,O1042,3.0,957.28,0.05,2727.35,1045.05,507.99,38.32,209.28,2728.25,2871.84,1045.12,104512.05,2010.08,994.10
66,O1067,6.0,271.04,0.20,1297.90,487.68,153.38,37.57,102.34,1300.99,1626.24,487.62,48762.10,1042.18,275.28
125,O1126,7.0,271.04,0.05,1795.73,647.93,153.38,36.08,185.39,1802.42,1897.28,647.90,64789.94,1456.96,536.68
140,O1141,13.0,5.99,0.10,69.87,27.60,3.35,39.50,6.81,70.08,77.87,27.60,2759.86,59.71,19.51
162,O1163,11.0,448.33,0.05,4662.63,1774.22,212.72,38.05,393.37,4685.05,4931.63,1774.13,177413.07,4056.54,1929.34
180,O1181,8.0,271.04,0.05,2059.87,741.08,153.38,36.08,185.39,2059.9,2168.32,743.20,74320.11,1721.10,647.44
213,O1214,7.0,271.04,0.10,1708.20,643.00,153.38,37.64,170.82,1707.55,1897.28,642.97,64296.65,1384.00,463.72
225,O1226,3.0,930.56,0.05,2651.09,954.39,529.62,36.00,265.11,2652.1,2791.68,954.39,95439.24,1856.36,797.12
237,O1238,6.0,271.04,0.05,1544.52,583.56,153.38,37.78,154.45,1544.93,1626.24,583.52,58351.97,1236.69,469.79
253,O1254,7.0,271.04,0.10,1708.20,620.00,153.38,36.29,170.82,1707.55,1897.28,619.91,61990.58,1384.00,463.72



Discount distribution:
Zero discount: 425
Non-zero discount: 160


In [21]:

# FINAL CLEANING STEP: Prepare files for MySQL

# Keep the original 30 columns as the source fields
original_cols = list(df_raw.columns)

# Use the 585 IKEA-style candidate records
final_df = clean_df[original_cols].copy()

# Standardise numeric and date types
numeric_cols = [
    "Quantity", "UnitPrice", "Discount",
    "Sales", "Profit", "Rating", "CostPrice",
    "Margin", "Tax"
]

for col in numeric_cols:
    final_df[col] = pd.to_numeric(
        final_df[col], errors="coerce"
    )

for col in ["OrderDate", "ShipDate", "DeliveryDate"]:
    final_df[col] = pd.to_datetime(
        final_df[col], errors="coerce"
    )

# Keep Month and Year consistent with OrderDate
final_df["Month"] = final_df["OrderDate"].dt.month.astype("Int64")
final_df["Year"] = final_df["OrderDate"].dt.year.astype("Int64")

# Add audit flags without overwriting source values
final_df["ExpectedSales"] = (
    final_df["Quantity"]
    * final_df["UnitPrice"]
    * (1 - final_df["Discount"])
).round(2)

final_df["SalesDifference"] = (
    final_df["Sales"] - final_df["ExpectedSales"]
).round(2)

final_df["SalesMismatchFlag"] = (
    final_df["SalesDifference"].abs() > 0.02
)

final_df["DeliveryBeforeOrderFlag"] = (
    final_df["DeliveryDate"] < final_df["OrderDate"]
)

final_df["DeliveryBeforeShipFlag"] = (
    final_df["DeliveryDate"] < final_df["ShipDate"]
)

# Use a consistent calculated margin for analysis.
# Keep the original Margin column for traceability.
final_df["CalculatedProfitMarginPct"] = (
    final_df["Profit"]
    .div(final_df["Sales"].replace(0, np.nan))
    .mul(100)
).round(2)

# Flag conflicting customer IDs, but do not invent replacements
name_counts = (
    final_df.groupby("CustomerID")["CustomerName"]
    .nunique()
)

final_df["CustomerIDConflictFlag"] = (
    final_df["CustomerID"].map(name_counts).gt(1)
)

# Prepare the separate generic-products review dataset
generic_review_df = df_generic_review[original_cols].copy()

# Build data quality summary
quality_summary = pd.DataFrame({
    "Metric": [
        "IKEA-style candidate records",
        "Generic model-labelled records set aside",
        "Missing values in original columns",
        "Duplicate OrderIDs",
        "Sales formula mismatches",
        "Delivery before order",
        "Delivery before shipping",
        "Rows affected by customer ID conflicts"
    ],
    "Value": [
        len(final_df),
        len(generic_review_df),
        int(final_df[original_cols].isna().sum().sum()),
        int(final_df["OrderID"].duplicated().sum()),
        int(final_df["SalesMismatchFlag"].sum()),
        int(final_df["DeliveryBeforeOrderFlag"].sum()),
        int(final_df["DeliveryBeforeShipFlag"].sum()),
        int(final_df["CustomerIDConflictFlag"].sum())
    ]
})

# Select flagged records for the audit workbook
sales_issues = final_df.loc[
    final_df["SalesMismatchFlag"]
].copy()

delivery_issues = final_df.loc[
    final_df["DeliveryBeforeOrderFlag"]
    | final_df["DeliveryBeforeShipFlag"]
].copy()

customer_issues = (
    final_df.groupby("CustomerID")
    .agg(
        RecordCount=("OrderID", "size"),
        UniqueNames=("CustomerName", "nunique"),
        CustomerNames=(
            "CustomerName",
            lambda x: " | ".join(sorted(x.unique()))
        )
    )
    .query("UniqueNames > 1")
    .reset_index()
)

# Export the main CSV for MySQL
final_df.to_csv(
    "ikea_style_candidate_cleaned.csv",
    index=False
)

# Keep the other records rather than losing them
generic_review_df.to_csv(
    "generic_products_for_review.csv",
    index=False
)

# Export the audit workbook
with pd.ExcelWriter(
    "ikea_data_quality_audit.xlsx",
    engine="openpyxl"
) as writer:
    final_df.to_excel(
        writer, sheet_name="Candidate_Cleaned", index=False
    )
    generic_review_df.to_excel(
        writer, sheet_name="Generic_Review", index=False
    )
    quality_summary.to_excel(
        writer, sheet_name="Quality_Summary", index=False
    )
    sales_issues.to_excel(
        writer, sheet_name="Sales_Mismatches", index=False
    )
    delivery_issues.to_excel(
        writer, sheet_name="Delivery_Issues", index=False
    )
    customer_issues.to_excel(
        writer, sheet_name="Customer_ID_Conflicts", index=False
    )

print("CLEANING AND EXPORT COMPLETED")
print("--------------------------------")
print("Main dataset:", final_df.shape)
print("Generic review dataset:", generic_review_df.shape)
print("\nQuality summary:")
display(quality_summary)

print("\nFiles saved:")
print("1. ikea_style_candidate_cleaned.csv")
print("2. generic_products_for_review.csv")
print("3. ikea_data_quality_audit.xlsx")


CLEANING AND EXPORT COMPLETED
--------------------------------
Main dataset: (585, 37)
Generic review dataset: (440, 30)

Quality summary:


,Metric,Value
0,IKEA-style candidate records,585
1,Generic model-labelled records set aside,440
2,Missing values in original columns,0
3,Duplicate OrderIDs,0
4,Sales formula mismatches,19
5,Delivery before order,0
6,Delivery before shipping,2
7,Rows affected by customer ID conflicts,567



Files saved:
1. ikea_style_candidate_cleaned.csv
2. generic_products_for_review.csv
3. ikea_data_quality_audit.xlsx
